# Clase 3 — Supremacía cuántica y simulación clásica

## Objetivos de aprendizaje

- Explicar en qué consiste la supremacía cuántica y cómo se verifica con el benchmarking de entropía cruzada (XEB)
- Usar la trotterización para aproximar la evolución temporal de un sistema cuántico
- Relacionar la profundidad de un circuito con la precisión de la simulación

In [ ]:
!pip install -q cirq tensorflow tensorflow-quantum

## Por qué importa

En 2019, Google anunció que su procesador Sycamore había resuelto en 200 segundos un cálculo que, según sus estimaciones, habría tardado miles de años en el superordenador clásico más potente del mundo. A ese hito lo llamamos **supremacía cuántica**: el momento en el que un ordenador cuántico resuelve una tarea que ningún ordenador clásico puede completar en un tiempo razonable. Si entiendes por qué ese cálculo es tan difícil de simular clásicamente, y cómo se comprueba que el resultado cuántico es correcto, tienes la intuición clave de todo este curso: la potencia de la computación cuántica crece de forma exponencial con el número de qubits, y el coste de simularla en un ordenador clásico crece igual de rápido.

## El coste exponencial de simular un circuito cuántico

Un sistema de n qubits en superposición necesita 2^n números para describirse por completo. Con 53 qubits, como los de Sycamore, obtienes más de 9.000 billones de números: ya no caben en la memoria de ningún ordenador clásico existente. Cuantos más qubits añades a tu circuito, más rápido se dispara el coste de simularlo, mientras el propio ordenador cuántico ejecuta ese circuito sin ese coste extra: literalmente, es ese sistema cuántico.

## Muestreo de circuitos aleatorios: la tarea que resolvió Sycamore

La tarea que demuestra la supremacía cuántica no es resolver un problema útil, sino algo más sutil: generar muestras de la distribución de probabilidad de salida de un circuito cuántico aleatorio y profundo. Un ordenador cuántico genera esas muestras de forma natural, con solo ejecutar el circuito una y otra vez. Un ordenador clásico, en cambio, tiene que calcular esa distribución multiplicando matrices gigantes, y ese cálculo se vuelve inviable a partir de cierto número de qubits y de profundidad de circuito.

## Cómo compruebas que el resultado cuántico es correcto: XEB

Si nadie puede calcular clásicamente la distribución de probabilidad completa, ¿cómo sabes que las muestras del ordenador cuántico son correctas? La respuesta es el benchmarking de entropía cruzada (XEB): en vez de calcular la distribución completa, calculas la probabilidad teórica de cada muestra concreta que observaste (mucho más barato que calcular todas las probabilidades posibles) y la comparas con lo que esperarías de un resultado puramente aleatorio. Cuanto más se parezcan tus muestras observadas a la distribución teórica del circuito, y no al ruido aleatorio, más confianza tienes en que el hardware funciona de verdad como un ordenador cuántico.

La forma que toma esa distribución teórica de salida —muchas probabilidades bajas y pocas altas, como manchas de luz láser dispersa— se conoce como distribución de Porter-Thomas, y es la huella dactilar que buscas al comparar resultados reales con la teoría. Una técnica relacionada, el benchmarking de pureza de las manchas (speckle purity benchmarking), te ayuda a separar cuánto de esa huella es señal cuántica real y cuánto es ruido del hardware.

## Cuantificando la confianza: significancia estadística

Afirmar que has demostrado supremacía cuántica es una afirmación fuerte, así que necesitas respaldarla con estadística rigurosa. Puedes usar herramientas como la prueba de Kolmogorov-Smirnov (KS), que compara si dos distribuciones de datos son estadísticamente indistinguibles, y técnicas de bootstrapping (remuestreo repetido de los mismos datos) para estimar cuánto varían tus conclusiones si repitieras el experimento. El resultado final se suele expresar en niveles sigma: cuantas más desviaciones estándar de margen tengas respecto a la hipótesis nula —que el resultado es ruido aleatorio—, más sólida es tu conclusión.

## La era NISQ: potentes, pero imperfectos

Sycamore, Bristlecone (su predecesor) y todos los procesadores cuánticos actuales pertenecen a la era NISQ (cuántico de escala intermedia y ruidoso, del capítulo dos): tienen decenas o unos pocos cientos de qubits, y el ruido limita la profundidad de los circuitos que puedes ejecutar de forma fiable. La supremacía cuántica demuestra que, incluso en esta era imperfecta, ya existen tareas que superan a la computación clásica. El reto de los próximos capítulos es distinto: no demostrar supremacía con circuitos aleatorios, sino usar esos mismos ordenadores ruidosos para resolver problemas útiles de machine learning, química y optimización.

## Trotterización: simular la evolución de un sistema cuántico

Una forma práctica de sentir el coste exponencial es simular la evolución en el tiempo de un sistema cuántico. Calcular esa evolución de forma exacta exige multiplicar matrices que crecen como 2^n, así que en la práctica la aproximas dividiendo la evolución en pasos pequeños: a esta técnica la llamamos trotterización. Cuantos más pasos de Trotter usas, más te acercas a la evolución exacta del sistema, pero también necesitas encadenar más puertas cuánticas, y recuerda el capítulo dos: más puertas significa más tiempo expuesto a la decoherencia. Este equilibrio entre precisión y profundidad de circuito es exactamente el mismo dilema al que se enfrenta cualquier ordenador cuántico NISQ real.

A continuación, aplicas esta idea a un ejemplo concreto: una cadena de espines de cuatro qubits con campo transverso, un sistema típico de física de materiales.

### Define el Hamiltoniano de la cadena de Ising con campo transverso

Cada qubit interactúa con su vecino mediante un término Z-Z, y además siente un campo transverso X propio.

In [ ]:
import cirq
import tensorflow_quantum as tfq
import numpy as np
import matplotlib.pyplot as plt

def cadena_ising_campo_transverso(n, campos):
    """Hamiltoniano de una cadena de n qubits con interacción Z-Z y campo X."""
    qubits = [cirq.GridQubit(0, i) for i in range(n)]
    hamiltoniano = cirq.PauliSum()
    for i in range(n - 1):
        hamiltoniano -= cirq.Z(qubits[i]) * cirq.Z(qubits[i + 1])
    for i in range(n):
        hamiltoniano -= campos[i] * cirq.X(qubits[i])
    return hamiltoniano

### Define la trotterización: un paso y la repetición completa

`paso_trotter` aplica cada término del Hamiltoniano por separado, como aproximación de un solo paso. `trotter_lie` encadena varios de esos pasos: cuantas más repeticiones usas, más fino es cada paso y más preciso el resultado final.

In [ ]:
def paso_trotter(hamiltoniano, t):
    """Aproxima exp(-i * H * t) aplicando cada término del Hamiltoniano por separado."""
    return cirq.Circuit([tfq.util.exponential([t * termino]) for termino in hamiltoniano])

def trotter_lie(hamiltoniano, t, repeticiones):
    """Cuantas más repeticiones, más fino el paso y más preciso el resultado."""
    circuito = cirq.Circuit()
    for _ in range(repeticiones):
        circuito += paso_trotter(hamiltoniano, t / repeticiones)
    return circuito

### Construye los circuitos y calcula la magnetización con TensorFlow Quantum

Comparas dos versiones del mismo experimento: una con un solo paso de Trotter (aproximación gruesa) y otra con diez pasos (aproximación fina). En ambos casos mides la magnetización total de la cadena, la suma de los operadores Z de cada qubit.

In [ ]:
n = 4
campos = [1.0] * n
hamiltoniano = cadena_ising_campo_transverso(n, campos)
qubits = [cirq.GridQubit(0, i) for i in range(n)]
magnetizacion = sum(cirq.Z(q) for q in qubits)

tiempos = np.linspace(0.01, 2, 20)

# Con solo 1 repetición, el paso de trotter es "grueso" y menos preciso
circuitos_1_paso = [trotter_lie(hamiltoniano, float(t), 1) for t in tiempos]
# Con 10 repeticiones, el paso es más fino y se acerca más a la evolución real
circuitos_10_pasos = [trotter_lie(hamiltoniano, float(t), 10) for t in tiempos]

capa_expectativa = tfq.layers.Expectation()
exp_1_paso = capa_expectativa(circuitos_1_paso, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()
exp_10_pasos = capa_expectativa(circuitos_10_pasos, symbol_names=[], symbol_values=[[] for _ in tiempos], operators=magnetizacion).numpy()

### Visualiza el resultado

Dibuja las dos curvas de magnetización y observa cuánto se separan entre sí a medida que pasa el tiempo.

In [ ]:
plt.plot(tiempos, exp_1_paso, label="1 paso de Trotter")
plt.plot(tiempos, exp_10_pasos, label="10 pasos de Trotter")
plt.xlabel("Tiempo")
plt.ylabel("Magnetización")
plt.legend()
plt.show()

La curva de un paso se separa cada vez más de la curva de referencia a medida que avanza el tiempo, porque el error de la aproximación se acumula en cada paso. La curva de diez pasos se mantiene mucho más cerca de la evolución real, al precio de un circuito con muchas más puertas encadenadas.

## Resumen de puntos clave

- Los investigadores demostraron la supremacía cuántica con una tarea de muestreo de circuitos aleatorios, inviable de simular en un ordenador clásico por el crecimiento exponencial (2^n) de los estados posibles
- El benchmarking de entropía cruzada (XEB) te permite verificar que las muestras cuánticas observadas corresponden a la distribución teórica del circuito (la distribución de Porter-Thomas) y no a ruido
- Cuantificas la confianza estadística en el resultado con pruebas como Kolmogorov-Smirnov, bootstrapping y niveles sigma
- Vivimos en la era NISQ: ordenadores cuánticos reales pero ruidosos, con pocos qubits y circuitos poco profundos
- La trotterización aproxima la evolución de un sistema cuántico dividiéndola en pasos pequeños: más pasos significan más precisión, pero también más profundidad de circuito y más exposición al ruido

## Glosario de acrónimos

- KS — prueba de Kolmogorov-Smirnov
- NISQ — cuántico de escala intermedia y ruidoso (Noisy Intermediate-Scale Quantum)
- XEB — benchmarking de entropía cruzada (Cross-Entropy Benchmarking)